In [57]:
import pandas as pd
pd.options.display.float_format = "{:,.0f}".format

df = pd.read_csv("../data/online_retail_II.csv", parse_dates=["InvoiceDate"])

In [4]:
# quick filter for now (full cleaning on Day 10)
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()
df["Revenue"] = df["Quantity"] * df["Price"]
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
Revenue               float64
dtype: object

In [5]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [ ]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")   # e.g. 2010-12
df["Weekday"] = df["InvoiceDate"].dt.day_name()          # Monday, Tuesday...
df["Hour"] = df["InvoiceDate"].dt.hour

df[["InvoiceDate", "YearMonth", "Weekday", "Hour"]].head()

,InvoiceDate,YearMonth,Weekday,Hour
0,2009-12-01 07:45:00,2009-12,Tuesday,7
1,2009-12-01 07:45:00,2009-12,Tuesday,7
2,2009-12-01 07:45:00,2009-12,Tuesday,7
3,2009-12-01 07:45:00,2009-12,Tuesday,7
4,2009-12-01 07:45:00,2009-12,Tuesday,7


In [ ]:
top5= df.groupby("Country")["Revenue"].sum().nlargest(5).index

In [16]:
pivot = pd.pivot_table(
    df[df["Country"].isin(top5)],
    index="YearMonth",
    columns="Country",
    values="Revenue",
    aggfunc="sum",
    fill_value="0",
)
pivot

Country,EIRE,France,Germany,Netherlands,United Kingdom
YearMonth,,,,,
2009-12,"20,100","6,522","10,097","15,205","750,317"
2010-01,"65,258","8,466","12,562","29,231","511,798"
2010-02,"21,539","9,024","9,124","17,255","455,136"
2010-03,"22,989","8,658","16,698","24,448","721,530"
2010-04,"23,929","7,643","23,683","6,889","584,610"
2010-05,"19,122","13,958","9,108","22,417","560,188"
2010-06,"22,934","9,173","15,679","24,225","652,412"
2010-07,"35,722","11,547","20,274","5,534","559,443"
2010-08,"26,006","16,570","9,943","25,029","596,550"


In [20]:
# summary 1: spend and orders per MonthYear
spend = (
    df.groupby("YearMonth")
    .agg(total_revenue=("Revenue", "sum"))
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
spend

,YearMonth,total_revenue
23,2011-11,"1,509,496"
11,2010-11,"1,470,272"
12,2010-12,"1,262,599"
10,2010-10,"1,165,484"
22,2011-10,"1,154,979"
21,2011-09,"1,058,590"
9,2010-09,"924,333"
3,2010-03,"833,570"
0,2009-12,"825,686"
17,2011-05,"770,536"


In [21]:
trend = spend.sort_values("YearMonth")
trend

,YearMonth,total_revenue
0,2009-12,"825,686"
1,2010-01,"652,709"
2,2010-02,"553,713"
3,2010-03,"833,570"
4,2010-04,"681,529"
5,2010-05,"659,859"
6,2010-06,"752,270"
7,2010-07,"650,713"
8,2010-08,"697,275"
9,2010-09,"924,333"


In [24]:
pivot = pd.pivot_table(
    df[df["Country"].isin(top5)],
    index="YearMonth",
    columns="Country",
    values="Revenue",
    aggfunc="sum",
    fill_value="0",
)
#pivot

In [56]:
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df.groupby("Weekday")["Revenue"].sum().reindex(order)

Weekday
Monday      3,645,130
Tuesday     4,185,863
Wednesday   3,601,790
Thursday    4,306,392
Friday      3,376,979
Saturday        9,803
Sunday      1,847,010
Name: Revenue, dtype: float64

In [29]:
revenue_hour= df.groupby("Hour")["Revenue"].sum()
revenue_hour

Hour
6            4
7       76,233
8      532,824
9    1,802,195
10   2,629,272
11   2,584,521
12   2,910,225
13   2,602,570
14   2,385,595
15   2,478,204
16   1,641,125
17     879,212
18     282,139
19     128,991
20      39,856
Name: Revenue, dtype: float64

In [ ]:
# summary 1: spend and orders per customer
spend_customer = df.groupby("Customer ID").agg(
    total_revenue=("Revenue", "sum"),
    n_invoices=("Invoice", "nunique"),
).reset_index()

In [36]:
# summary 2: first and last purchase date
dates = df.groupby("Customer ID").agg(
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max"),
).reset_index()


In [ ]:
# summary 3: one country per customer
country = df[["Customer ID", "Country"]].drop_duplicates("Customer ID")

customers = spend_customer.merge(dates, on="Customer ID", how="left") \
                 .merge(country, on="Customer ID", how="left")

print(len(spend), len(customers))   # should be equal
customers.head()

25 5878


,Customer ID,total_revenue,n_invoices,first_purchase,last_purchase,Country
0,"12,346","77,556",12,2009-12-14 08:34:00,2011-01-18 10:01:00,United Kingdom
1,"12,347","5,633",8,2010-10-31 14:20:00,2011-12-07 15:52:00,Iceland
2,"12,348","2,019",5,2010-09-27 14:59:00,2011-09-25 13:13:00,Finland
3,"12,349","4,429",4,2010-04-29 13:20:00,2011-11-21 09:51:00,Italy
4,"12,350",334,1,2011-02-02 16:01:00,2011-02-02 16:01:00,Norway


In [53]:
sl_khach = (customers["n_invoices"] == 1).sum()
sl_khach

np.int64(1623)

In [54]:
  print((customers["n_invoices"] == 1).mean() * 100)

27.611432460020414


In [44]:
top10= (
        customers.sort_values("total_revenue",ascending = False).head(10)[["Customer ID","total_revenue","n_invoices","first_purchase","Country"]]
                    .reset_index(drop=True)
)
top10

,Customer ID,total_revenue,n_invoices,first_purchase,Country
0,"18,102","608,822",145,2009-12-01 09:24:00,United Kingdom
1,"14,646","528,603",151,2009-12-02 16:52:00,Netherlands
2,"14,156","313,946",156,2009-12-01 12:30:00,EIRE
3,"14,911","295,973",398,2009-12-01 11:41:00,EIRE
4,"17,450","246,973",51,2010-09-27 16:59:00,United Kingdom
5,"13,694","196,483",143,2009-12-04 15:26:00,United Kingdom
6,"17,511","175,604",60,2009-12-02 10:52:00,United Kingdom
7,"16,446","168,472",2,2011-05-18 09:52:00,United Kingdom
8,"16,684","147,143",55,2009-12-07 12:56:00,United Kingdom
9,"12,415","144,458",28,2010-06-30 08:30:00,Australia


In [55]:
  compare.sort_values("total_revenue", ascending=False).head(3)
  compare.sort_values("n_customers", ascending=False).head(3)

,YearMonth,n_customers,total_revenue,rev_per_customer
23,2011-11,1664,"1,509,496",907
11,2010-11,1607,"1,470,272",915
10,2010-10,1497,"1,165,484",779
